## Import Libraries

In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.10.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Load Preprocessed Data

In [ ]:
from pathlib import Path

dataset_path = Path("../../dataset")
dl_data_path = dataset_path / "dl_data"

X_train = np.load(dl_data_path / "X_train.npy")
X_validation = np.load(dl_data_path / "X_validation.npy")
X_test = np.load(dl_data_path / "X_test.npy")

y_train = np.load(dl_data_path / "y_train.npy")
y_validation = np.load(dl_data_path / "y_validation.npy")
y_test = np.load(dl_data_path / "y_test.npy")

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_validation:", y_validation.shape)
print("y_test:", y_test.shape)

X_train: (5969, 9000)
X_validation: (1279, 9000)
X_test: (1280, 9000)
y_train: (5969,)
y_validation: (1279,)
y_test: (1280,)


## Check Labels

In [3]:
print("Unique training labels:", np.unique(y_train))
print("Unique validation labels:", np.unique(y_validation))
print("Unique test labels:", np.unique(y_test))

Unique training labels: [0 1 2 3]
Unique validation labels: [0 1 2 3]
Unique test labels: [0 1 2 3]


## Add Channel Dimension

In [12]:
X_train = X_train[..., np.newaxis]
X_validation = X_validation[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("\n===== XCEPTION INPUT SHAPES =====")

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)


===== XCEPTION INPUT SHAPES =====
X_train: (5969, 9000, 1)
X_validation: (1279, 9000, 1)
X_test: (1280, 9000, 1)


## Residual Block

In [13]:
def residual_block(x, filters, kernel_size=7, stride=1):
    shortcut = x

    # Main path
    x = layers.Conv1D(
        filters,
        kernel_size,
        strides=stride,
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv1D(
        filters,
        kernel_size,
        strides=1,
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)

    # Shortcut projection if shape changes
    if shortcut.shape[-1] != filters or stride != 1:
        shortcut = layers.Conv1D(
            filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)

    return x

## Transformer Encoder Block

In [14]:
def transformer_encoder(
    x,
    num_heads=4,
    key_dim=32,
    ff_dim=128,
    dropout=0.1
):
    # Multi-Head Self Attention
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout
    )(x, x)

    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    # Feed Forward Network
    ff_output = layers.Dense(ff_dim, activation="relu")(x)
    ff_output = layers.Dropout(dropout)(ff_output)
    ff_output = layers.Dense(x.shape[-1])(ff_output)

    x = layers.Add()([x, ff_output])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    return x

## Build Hybrid CNN + Residual + Attention + Transformer

In [15]:
def build_hybrid_cnn_transformer(
    input_shape=(9000, 1),
    num_classes=4
):
    inputs = layers.Input(shape=input_shape)

    # =========================================================
    # Initial CNN feature extraction
    # =========================================================

    x = layers.Conv1D(
        32,
        kernel_size=15,
        strides=2,
        padding="same",
        use_bias=False
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Downsample:
    # 9000 -> ~4500
    x = layers.MaxPooling1D(
        pool_size=3,
        strides=2,
        padding="same"
    )(x)

    # =========================================================
    # Residual CNN blocks
    # =========================================================

    # ~4500 -> ~2250
    x = residual_block(
        x,
        filters=64,
        kernel_size=7,
        stride=2
    )

    # ~2250 -> ~1125
    x = residual_block(
        x,
        filters=128,
        kernel_size=7,
        stride=2
    )

    # Keep sequence length
    x = residual_block(
        x,
        filters=128,
        kernel_size=5,
        stride=1
    )

    # =========================================================
    # Projection before Transformer
    # =========================================================

    x = layers.Conv1D(
        128,
        kernel_size=1,
        padding="same"
    )(x)

    # =========================================================
    # Transformer Encoder
    # =========================================================

    x = transformer_encoder(
        x,
        num_heads=4,
        key_dim=32,
        ff_dim=256,
        dropout=0.1
    )

    # Second Transformer block
    x = transformer_encoder(
        x,
        num_heads=4,
        key_dim=32,
        ff_dim=256,
        dropout=0.1
    )

    # =========================================================
    # Classification head
    # =========================================================

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(
        128,
        activation="relu"
    )(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="Hybrid_CNN_Residual_Attention_Transformer"
    )

    return model

## Create Model

In [16]:
model = build_hybrid_cnn_transformer(
    input_shape=X_train.shape[1:],
    num_classes=4
)

model.summary()

Model: "Hybrid_CNN_Residual_Attention_Transformer"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 9000, 1)]    0           []                               
                                                                                                  
 conv1d_2 (Conv1D)              (None, 4500, 32)     480         ['input_3[0][0]']                
                                                                                                  
 batch_normalization (BatchNorm  (None, 4500, 32)    128         ['conv1d_2[0][0]']               
 alization)                                                                                       
                                                                                                  
 activation (Activation)        (None, 4500, 32)     0    

## Compile Model

In [17]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "Hybrid_CNN_Residual_Attention_Transformer"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 9000, 1)]    0           []                               
                                                                                                  
 conv1d_2 (Conv1D)              (None, 4500, 32)     480         ['input_3[0][0]']                
                                                                                                  
 batch_normalization (BatchNorm  (None, 4500, 32)    128         ['conv1d_2[0][0]']               
 alization)                                                                                       
                                                                                                  
 activation (Activation)        (None, 4500, 32)     0    

 conv1d_3 (Conv1D)              (None, 1125, 64)     14336       ['max_pooling1d[0][0]']          
                                                                                                  
 batch_normalization_1 (BatchNo  (None, 1125, 64)    256         ['conv1d_3[0][0]']               
 rmalization)                                                                                     
                                                                                                  
 activation_1 (Activation)      (None, 1125, 64)     0           ['batch_normalization_1[0][0]']  
                                                                                                  
 conv1d_4 (Conv1D)              (None, 1125, 64)     28672       ['activation_1[0][0]']           
                                                                                                  
 conv1d_5 (Conv1D)              (None, 1125, 64)     2048        ['max_pooling1d[0][0]']          
          

## Callbacks

In [18]:
checkpoint_path = "best_hybrid_cnn_transformer.keras"

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    ModelCheckpoint(
        checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

## Train

In [19]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    epochs=50,
    batch_size=8,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
747/747 [==============================] - ETA: 0s - loss: 1.0248 - accuracy: 0.5778
Epoch 1: val_loss improved from inf to 1.03818, saving model to best_hybrid_cnn_transformer.keras
747/747 [==============================] - 71s 77ms/step - loss: 1.0248 - accuracy: 0.5778 - val_loss: 1.0382 - val_accuracy: 0.5950
Epoch 2/50
746/747 [============================>.] - ETA: 0s - loss: 0.9992 - accuracy: 0.5927
Epoch 2: val_loss improved from 1.03818 to 0.99376, saving model to best_hybrid_cnn_transformer.keras
747/747 [==============================] - 56s 75ms/step - loss: 0.9992 - accuracy: 0.5927 - val_loss: 0.9938 - val_accuracy: 0.5950
Epoch 3/50
746/747 [============================>.] - ETA: 0s - loss: 1.0062 - accuracy: 0.5952
Epoch 3: val_loss improved from 0.99376 to 0.98942, saving model to best_hybrid_cnn_transformer.keras
747/747 [==============================] - 57s 76ms/step - loss: 1.0062 - accuracy: 0.5951 - val_loss: 0.9894 - val_accuracy: 0.5950
Epoch 4/50


## Test Loss & Accuracy

In [20]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

40/40 [==============================] - 3s 73ms/step - loss: 0.9831 - accuracy: 0.5953
Test Loss: 0.9831
Test Accuracy: 0.5953


## Predictions

In [ ]:
y_pred_prob = model.predict(
    X_test,
    batch_size=8,
    verbose=1
)

y_pred = np.argmax(
    y_pred_prob,
    axis=1
)

print("Predictions shape:", y_pred.shape)

## Classification Report

In [ ]:
class_names = ["N", "A", "O", "~"]

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4
    )
)

## Macro / Weighted Metrics

In [ ]:
macro_precision = precision_score(
    y_test,
    y_pred,
    average="macro"
)

macro_recall = recall_score(
    y_test,
    y_pred,
    average="macro"
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall:    {macro_recall:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
print(f"Weighted F1:     {weighted_f1:.4f}")

## F1 Classes

In [ ]:
class_f1 = f1_score(
    y_test,
    y_pred,
    average=None
)

for class_name, score in zip(class_names, class_f1):
    print(f"F1-{class_name}: {score:.4f}")

## Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

## Training Curves

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Train Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")

plt.legend()
plt.grid(True)

plt.show()

## Accuracy Curve

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Train Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")

plt.legend()
plt.grid(True)

plt.show()